# atlm_pro MP1 — Continued Pretraining of SmolLM2-360M

**Goal.** Continue the **pretraining** of the base `SmolLM2-360M` language
model on a corpus of raw job-description text, adapting it to the job-postings
domain — plain next-token prediction, no labels and no queries (turning the
model into a recruiter-query assistant is MP2's job).

**Optimization experiment.** The required knob is **full fine-tuning vs. LoRA**:
two controlled runs with everything else held constant, compared by perplexity
and sample generations.

This is the **consolidated MP1 notebook** — business context, dataset
understanding, corpus construction, the continued-pretraining experiment and
results — and the single source for the MP1 report.

| Section | Scope |
|---|---|
| 1. Business Understanding | why job-description quality matters — cost, regulation, evidence |
| 2. Work Plan | the three-assignment pipeline and the datasets |
| 3. Setup | imports, paths |
| 4. Load datasets | read files, shape summary |
| 5. Djinni — the core dataset | schema, distributions, text quality, templates |
| 6. LinkedIn — out-of-domain test set | schema, text quality, families, metadata |
| 7. Comparative analysis | Djinni vs LinkedIn, side by side |
| 8. Prompt-readiness & recommendations | training implications |
| 9. Skill → Role mapping | do skills associate consistently with roles? |
| 10. Tokenisation analysis | token lengths, technical-term fragmentation |
| 11. Semantic embedding analysis | do descriptions form coherent clusters? |
| 12. Building the corpus | apply the EDA decisions → fixed train/val/test corpus |
| 13. Continued pretraining | full fine-tuning vs LoRA |
| 14. Results | loss curves, perplexity, generations, 135M-vs-360M comparison |
| 15. Interactive testing | feed the trained model your own prompts |

Two datasets are used: **Djinni** (real IT job postings — train/val/in-domain
test) and **LinkedIn** (cross-industry — out-of-domain test only).

Part **MP1** of the three-part project: **MP1 continued pretraining** → MP2
alignment → Final system.

## 1. Business Understanding

### 1.1 The Scale of the Problem

Hiring is among the most resource-intensive processes in any organization, and a
significant share of that cost traces directly to the recruitment pipeline
itself, job advertising, recruiter time, applicant screening, and interview
coordination, rather than the employee's eventual compensation. In the United
States, the SHRM 2025 Talent Acquisition Benchmarking Report places the average
recruitment cost per hire at \$5,475 for non-executive roles and \$35,879 for
executive positions, representing a 21% increase from 2022 [shrm2025]. In the
United Kingdom, the Chartered Institute of Personnel and Development (CIPD)
estimates the average recruitment cost per hire at £6,125, encompassing internal
staff time and external fees such as job advertising and agency charges
[cipd2025]. The scale of the recruitment market is substantial: the EURES portal
alone publishes approximately 60 million job vacancies per year across Europe
[eures2024], while LinkedIn hosts roughly 15.7 million active job listings
globally [novoresume2026]. Each unfilled vacancy costs organizations an estimated
\$500 per day in lost productivity [shrm2025], and the average time to fill a
position ranges from 36 to 42 days [recruitee2025]. At this volume, even small
inefficiencies in the hiring funnel compound into substantial financial waste.

### 1.2 Where the Problem Is Most Acute

Recruitment cost and difficulty vary dramatically by sector. Technology companies
spent \$6,000-\$8,000 per hire in 2023, while healthcare organizations faced
recruitment costs of \$9,000-\$12,000 due to credentialing, regulatory
compliance, and acute talent shortages [engagedly2025]. Technical roles such as
engineering positions can reach \$10,000-\$20,000 per hire [timeclick2026]. In
Europe, the EURES 2024 Report on Labour Shortages and Surpluses confirms that
severe shortages persist in highly skilled professions as healthcare,
engineering, and IT, as well as in trades such as welding, electrical work, and
construction [eures2024]. All 31 EURES countries reported shortage occupations
across nearly all occupational categories, with Malta, Slovakia, Bulgaria, Italy,
and Romania identifying the greatest numbers [eures2024]. These are precisely the
sectors where job descriptions demand the most precision, niche skill
combinations, regulatory language, and accurate seniority calibration, and where
vague or generic postings fail most visibly.

### 1.3 The Regulatory Landscape

The job description challenge is compounded by an evolving regulatory environment
on both sides of the Atlantic. In the United States, multiple states, including
California, Colorado, New York, and Massachusetts, have enacted pay transparency
laws requiring salary ranges in job postings [vbeyond2025]. In Europe, the
regulatory pressure is even more significant. The EU Pay Transparency Directive
(Directive (EU) 2023/970), which must be transposed into national law by 7 June
2026, requires all employers across all 27 member states to include salary ranges
or starting pay levels in job postings, use gender-neutral job titles and
descriptions, and refrain from asking candidates about their salary history
[ogletree2026]. Vague formulations such as "competitive salary" or "negotiable"
will no longer satisfy legal requirements [salarytransparency2026]. Some member
states, including Ireland and the Netherlands, are adopting stricter
interpretations that mandate salary ranges directly in the advertisement itself
[salarytransparency2026]. This regulatory shift transforms job description
quality from a best practice into a compliance obligation, making structured,
template-aware generation tools not merely convenient but operationally necessary
for organizations hiring across European markets.

### 1.4 What Makes a Good Job Description

Research consistently demonstrates that job description quality directly
determines hiring outcomes and defines the criteria our system must enforce.
According to Indeed, 52% of job seekers report that the quality of a job
description is "very" or "extremely" influential in their decision to apply, and
63% of candidates have declined to apply because they did not understand the
required skills or tools [indeed2024]. On the supply side, up to 42% of received
resumes do not meet basic job requirements [sullivan2020], a problem driven by
vague or inflated postings. Organizations that publish "roles with unclear
requirements and inflated expectations" directly contribute to talent shortages
[isolved2026]. Inclusive language has measurable impact: job descriptions written
with gender-neutral language receive up to 42% more responses [compono2025], and
61% of candidates identify the salary range as the single most important element
in a job posting [linkedin2025]. According to Glassdoor, 70% of applicants want
salary information before taking any further step in the application process
[glassdoor2023]. These findings define the quality criteria that generated job
descriptions must enforce: structural completeness (job title, role summary,
responsibilities, required skills, qualifications, and benefits), salary
transparency with realistic ranges, specific and calibrated skill requirements,
inclusive and gender-neutral language, and consistent seniority alignment
throughout the posting.

## 2. Work Plan

This project develops a domain-adapted language model that generates structured,
high-quality job postings from minimal input — a role title, required skills,
seniority level, and organizational constraints. Rather than relying on generic
large language models, we build a specialized three-stage pipeline where each
assignment contributes a distinct capability. The full arc proceeds from domain
adaptation through alignment to a production-ready system.

Two publicly available job posting corpora serve as the foundation for all three
stages. The first is the Djinni Recruitment Dataset [djinni2024], available on
HuggingFace, containing approximately 142,000 English language job descriptions
from the Djinni IT job platform (2020–2023), with structured fields including
position title, full description, company name, experience requirements, primary
keyword, and English proficiency level. The second is the LinkedIn Job Postings
Dataset [kagglelinkedin], available on Kaggle, containing approximately 33,000
job postings with structured metadata including title, company, description,
salary range, location, and application details. Before any training begins, a
dedicated Data Understanding phase will explore both datasets, examining
distributions of job categories, seniority levels, description lengths,
vocabulary coverage, and structural patterns, to inform preprocessing decisions
and identify potential quality issues, class imbalances, or domain biases.

### 2.1 Assignment 1 — Domain Adaptation

**Purpose.** The goal of the first stage is to teach a pretrained language model
what job postings sound like, their vocabulary, structural conventions, section
ordering, and domain-specific phrasing, through continued pretraining on raw job
description text. The model's next-token prediction objective remains unchanged;
only the data distribution shifts toward the recruitment domain. At the end of
this stage, the model should exhibit measurably lower perplexity on held-out job
posting text compared to the base checkpoint.

**Tools and method.** A small open-source pretrained model (SmolLM2-360M)
suitable for training on limited hardware was selected. The combined and cleaned
text from both datasets is used as the continued pretraining corpus (Por
verificar se se vai utilizar todo ou parte do data-set). Training is conducted
using HuggingFace Transformers and the Trainer API, with — TBC. Evaluation relies
on perplexity on a held-out split and qualitative inspection of prompted text
completions (TBC).

### 2.2 Assignment 2 — Alignment

**Purpose.** The domain-adapted model from Assignment 1 can generate text that
resembles job postings but cannot follow specific instructions (e.g., "write a
posting for a senior Python developer, remote, fintech"). The second stage adds
two training phases: supervised fine-tuning (SFT) on instruction-response pairs
teaches the model to produce job descriptions on command, and preference
alignment via Direct Preference Optimization (DPO) teaches it to prefer responses
that respect stated constraints, use inclusive language, maintain structural
completeness, and calibrate seniority consistently.

**Tools and method.** SFT and DPO training are implemented using the HuggingFace
TRL library (SFTTrainer and DPOTrainer). The SFT dataset (800-1,000
instruction-response pairs TBC) is constructed from the existing job postings by
pairing each posting with a synthetic instruction that would have produced it.
The DPO preference dataset (400-500 prompt, chosen and rejected triplets TBC) is
generated by sampling multiple candidate outputs from the SFT model and ranking
them using a stronger model as judge (RLAIF). Evaluation is a three-way
comparison — base pretrained model, domain-adapted model, and aligned model, on
the same fixed prompt set, using both automatic metrics (perplexity,
LLM-as-judge win-rate) and qualitative analysis of constraint satisfaction and
structural quality.

### 2.3 Assignment 3 — System Integration

**Purpose.** The final stage wraps the aligned model into a functional system
that addresses the remaining gaps: grounding in real market data and automated
quality assurance. Two additional components are integrated on top of the
fine-tuned model to form a complete job description generation pipeline.

**Tools and method.** The first component is Retrieval-Augmented Generation
(RAG): a vector store is built over the original job posting corpora, indexed by
role, industry, and skill set. When a user requests a posting, semantically
similar real postings are retrieved and provided as context, grounding the
model's output in actual market conventions, realistic salary ranges, standard
qualification phrasing, and industry-specific requirements. The second component
is an LLM-as-Judge evaluation loop with iterative refinement: after generating a
draft posting, a structured rubric automatically checks for section completeness,
skill-prompt alignment, seniority consistency, and inclusive language. If the
rubric score falls below a threshold, the critique is fed back to the model for
regeneration. The system is exposed through a functional entry point (CLI,
notebook, or simple UI). Evaluation compares the full system against the
Assignment 2 aligned model and the unmodified base model on 30-50 test prompts,
measuring constraint satisfaction rate, structural completeness, and qualitative
output quality.

## 3. Setup

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_colwidth", 120)

# Locate the project root (the folder that contains data/).
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
JOBS = PROJECT_ROOT / "data" / "jobs"

print("Project root :", PROJECT_ROOT)
print("Jobs data dir:", JOBS)
assert JOBS.exists(), f"data/jobs not found starting from {Path.cwd()}"

## 4. Load datasets

Both datasets are loaded here and a top-level shape summary is produced.
All exploratory analysis follows in the dedicated sections below.


In [ ]:
# ── Djinni ───────────────────────────────────────────────────────────────────
djinni = pd.read_parquet(JOBS / "djinni" / "train-00000-of-00001.parquet")
djinni = djinni.drop(columns=["__index_level_0__"], errors="ignore")

# ── LinkedIn ─────────────────────────────────────────────────────────────────
linkedin_keep = ["job_id", "title", "description", "company_name", "location",
                 "formatted_work_type", "formatted_experience_level", "skills_desc"]
linkedin = pd.read_csv(JOBS / "linkedin" / "postings.csv", usecols=linkedin_keep)

# ── Summary ──────────────────────────────────────────────────────────────────
datasets = {"Djinni": djinni, "LinkedIn": linkedin}

pd.DataFrame({
    "Dataset": list(datasets.keys()),
    "Rows":    [df.shape[0] for df in datasets.values()],
    "Columns": [df.shape[1] for df in datasets.values()],
})


## 5. Djinni - the core dataset

[`lang-uk/recruitment-dataset-job-descriptions-english`](https://huggingface.co/datasets/lang-uk/recruitment-dataset-job-descriptions-english)
— real IT job postings scraped from the Djinni platform (2020–2023).

This is the **core training dataset**. The analysis below covers:
- schema & missing values
- primary keyword (tech domain) distribution
- target text (`Long Description`) quality and length
- structured prompt features: experience level, English level
- description length vs. seniority
- repetition / template detection


### 5.1 Schema and missing values

In [ ]:
print(f"Shape: {djinni.shape[0]:,} rows × {djinni.shape[1]} columns\n")

schema = pd.DataFrame({
    "dtype":      [djinni[c].dtype for c in djinni.columns],
    "non_null":   [djinni[c].notna().sum() for c in djinni.columns],
    "missing_%":  [(djinni[c].isna().mean() * 100).round(2) for c in djinni.columns],
    "n_unique":   [djinni[c].nunique() for c in djinni.columns],
}, index=djinni.columns)

display(schema)


### 5.2 Primary keyword (tech domain) distribution

`Primary Keyword` is the main structured signal for prompting:
it labels the tech stack or domain of each posting.


In [ ]:
kw = djinni["Primary Keyword"].value_counts()
print(f"{kw.size} distinct primary keywords\n")

ax = kw.head(20).iloc[::-1].plot.barh(
    figsize=(8, 5), title="Djinni — top 20 primary keywords")
ax.set_xlabel("postings")
plt.tight_layout()
plt.show()

# Full list for reference
display(kw.reset_index().rename(columns={"Primary Keyword": "keyword", "count": "postings"}))


### 5.3 Target text quality (`Long Description`)

Key questions: How many non-null, unique descriptions are there?
What is the length distribution?


In [ ]:
text = djinni["Long Description"].dropna().astype(str)

print(f"Non-null descriptions : {len(text):,}  ({len(text)/len(djinni):.1%})")
print(f"Unique descriptions   : {text.nunique():,}  ({text.nunique()/len(text):.1%})")
print(f"Duplicate ratio       : {1 - text.nunique()/len(text):.3f}\n")
print("Length in characters:")
print(text.str.len().describe().round(0))

ax = text.str.len().clip(upper=8000).plot.hist(
    bins=60, figsize=(8, 3),
    title="Djinni — Long Description length (chars, clipped at 8 000)")
ax.set_xlabel("characters")
plt.tight_layout()
plt.show()


### 5.4 Structured prompt features: experience level and English level

`Exp Years` and `English Level` are the two categorical fields that can be used
alongside `Primary Keyword` to build richer prompts.
This section characterises their distributions and coverage in one place.


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# Clean values
djinni["Exp Years_clean"] = djinni["Exp Years"].astype(str).str.strip()
djinni["English Level_clean"] = djinni["English Level"].astype(str).str.strip()

# Remove fake nan strings
djinni.loc[djinni["Exp Years"].isna(), "Exp Years_clean"] = pd.NA
djinni.loc[djinni["English Level"].isna(), "English Level_clean"] = pd.NA

# Coverage
cov = pd.DataFrame({
    "Feature": ["Exp Years", "English Level"],
    "Non-null": [
        djinni["Exp Years"].notna().sum(),
        djinni["English Level"].notna().sum()
    ],
    "Non-null %": [
        round(djinni["Exp Years"].notna().mean() * 100, 2),
        round(djinni["English Level"].notna().mean() * 100, 2)
    ],
    "N distinct": [
        djinni["Exp Years_clean"].nunique(dropna=True),
        djinni["English Level_clean"].nunique(dropna=True)
    ],
    "Values": [
        ", ".join(djinni["Exp Years_clean"].dropna().unique().astype(str)),
        ", ".join(djinni["English Level_clean"].dropna().unique().astype(str))
    ]
})

display(cov)

# Real distributions
exp_counts = djinni["Exp Years_clean"].value_counts(dropna=False)
eng_counts = djinni["English Level_clean"].value_counts(dropna=False)

display(exp_counts)
display(eng_counts)

# Side-by-side bar charts
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

if not exp_counts.empty:
    exp_counts.plot.bar(ax=axes[0], rot=30)
    axes[0].set_title("Experience level distribution")
    axes[0].set_xlabel("")
    axes[0].set_ylabel("postings")
else:
    axes[0].text(0.5, 0.5, "No Exp Years data", ha="center", va="center")
    axes[0].set_axis_off()

if not eng_counts.empty:
    eng_counts.plot.bar(ax=axes[1], rot=30)
    axes[1].set_title("English level distribution")
    axes[1].set_xlabel("")
    axes[1].set_ylabel("postings")
else:
    axes[1].text(0.5, 0.5, "No English Level data", ha="center", va="center")
    axes[1].set_axis_off()

plt.suptitle("Djinni — structured prompt features", y=1.01)
plt.tight_layout()
plt.show()

# Cross-tab
ct = pd.crosstab(
    djinni["Exp Years_clean"],
    djinni["English Level_clean"],
    normalize="index"
).round(3)

if ct.empty:
    print("Cross-tab is empty. There is no overlap between Exp Years and English Level.")
else:
    ax = ct.plot.bar(
        figsize=(11, 4),
        stacked=True,
        title="Djinni — English level share per experience tier"
    )
    ax.set_xlabel("Experience required")
    ax.set_ylabel("proportion")
    ax.legend(title="English level", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.show()

    display(ct)

### 5.5 Description length vs. seniority

Does experience level influence how verbose a job description is?
This is relevant because the model should calibrate output length to the prompt.


In [ ]:
djinni["desc_len"] = djinni["Long Description"].astype(str).str.len()
djinni["Exp Years_clean"] = djinni["Exp Years"].astype(str).str.strip()

# Usar os valores reais existentes no dataset
present_exp = djinni["Exp Years_clean"].dropna().value_counts().index.tolist()

box_data = []

for e in present_exp:
    values = (
        djinni.loc[djinni["Exp Years_clean"] == e, "desc_len"]
        .clip(upper=8000)
        .dropna()
        .to_numpy()
    )
    
    if len(values) > 0:
        box_data.append(values)

if len(box_data) == 0:
    print("No valid data available for boxplot.")
    display(djinni["Exp Years"].value_counts(dropna=False))
else:
    fig, ax = plt.subplots(figsize=(9, 4))

    ax.boxplot(
        box_data,
        tick_labels=present_exp,
        notch=True,
        patch_artist=True
    )

    ax.set_title("Djinni — description length by experience level")
    ax.set_xlabel("Experience required")
    ax.set_ylabel("characters (clipped at 8 000)")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))

    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()

    exp_stats = (
        djinni[djinni["Exp Years_clean"].isin(present_exp)]
        .groupby("Exp Years_clean")["desc_len"]
        .agg(n="count", mean="mean", median="median")
        .round(0)
        .reindex(present_exp)
    )

    display(exp_stats)

### 5.6 Repetition and template detection

Does the same keyword always produce the same description, or is there genuine
variety per tech domain?
A low diversity score (≈1 distinct description per keyword) would indicate
a lookup-table dataset rather than a true language-generation resource.


In [ ]:
diversity = (
    djinni.groupby("Primary Keyword")["Long Description"]
    .nunique()
    .sort_values(ascending=False)
    .reset_index(name="distinct_descriptions")
)

print("Distinct descriptions per keyword — summary:")
display(diversity.describe().round(0))
print("\nTop 15 keywords by description variety:")
display(diversity.head(15))


## 6. LinkedIn - out-of-domain test set

[Kaggle: LinkedIn Job Postings 2023–2024](https://www.kaggle.com/datasets/arshkon/linkedin-job-postings)
— real, cross-industry postings.

This dataset is held out **entirely from training** and used only to measure
how well an IT-trained model generalises to other domains.

The analysis below covers:
- schema & missing values
- target text (`description`) quality and length
- job title and industry distribution
- structured metadata coverage
- repetition / template detection


### 6.1 Schema and missing values

In [ ]:
print(f"Shape: {linkedin.shape[0]:,} rows × {linkedin.shape[1]} columns\n")

schema_li = pd.DataFrame({
    "dtype":     [linkedin[c].dtype for c in linkedin.columns],
    "non_null":  [linkedin[c].notna().sum() for c in linkedin.columns],
    "missing_%": [(linkedin[c].isna().mean() * 100).round(2) for c in linkedin.columns],
    "n_unique":  [linkedin[c].nunique() for c in linkedin.columns],
}, index=linkedin.columns)

display(schema_li)


### 6.2 Target text quality (`description`)


In [ ]:
li_text = linkedin["description"].dropna().astype(str)

print(f"Non-null descriptions : {len(li_text):,}  ({len(li_text)/len(linkedin):.1%})")
print(f"Unique descriptions   : {li_text.nunique():,}  ({li_text.nunique()/len(li_text):.1%})")
print(f"Duplicate ratio       : {1 - li_text.nunique()/len(li_text):.3f}\n")
print("Length in characters:")
print(li_text.str.len().describe().round(0))

ax = li_text.str.len().clip(upper=8000).plot.hist(
    bins=60, figsize=(8, 3),
    title="LinkedIn — description length (chars, clipped at 8 000)")
ax.set_xlabel("characters")
plt.tight_layout()
plt.show()


### 6.3 Job family distribution

LinkedIn spans all industries. A keyword-based taxonomy groups job titles into
broad families to reveal how balanced (or skewed) the out-of-domain test set is.


In [ ]:
import re

family_map = {
    "Engineering / Dev":  r"engineer|developer|software|backend|frontend|fullstack|devops|sre|cloud",
    "Data / AI / ML":     r"data |machine learning|analyst|scientist|analytics|\bbi\b|nlp|llm|\bai\b",
    "Design / UX":        r"design|ux |ui |product design|creative",
    "Marketing":          r"market|seo|growth|brand|content|social media|copywriter",
    "Sales / BD":         r"sales|account exec|business dev|account manager",
    "Finance":            r"finance|accountant|financi|controller|audit|\btax\b",
    "HR / Recruiting":    r"recruiter|talent|human resource|\bhr\b|people ops",
    "Operations / PM":    r"operat|product manager|project manager|scrum|agile|supply",
    "Customer Success":   r"customer success|support|\bcx\b|service desk|helpdesk",
    "Legal / Compliance": r"legal|counsel|compliance|attorney|paralegal",
}

def classify_title(title):
    t = str(title).lower()
    for family, pattern in family_map.items():
        if re.search(pattern, t):
            return family
    return "Other"

linkedin["job_family"] = linkedin["title"].map(classify_title)
family_counts = linkedin["job_family"].value_counts()

ax = family_counts.sort_values().plot.barh(
    figsize=(9, 5), color="steelblue",
    title="LinkedIn — job family distribution")
ax.set_xlabel("postings")
plt.tight_layout()
plt.show()

display(family_counts.reset_index().rename(columns={"job_family": "family", "count": "postings"}))


### 6.4 Structured metadata coverage

Which LinkedIn fields can realistically be used as prompt inputs or evaluation
filters?


In [ ]:
meta_cols = ["title", "formatted_work_type", "formatted_experience_level",
             "location", "company_name", "skills_desc"]

meta = pd.DataFrame({
    "Feature":    meta_cols,
    "Non-null %": [(linkedin[c].notna().mean() * 100).round(2) for c in meta_cols],
    "N distinct": [linkedin[c].nunique() for c in meta_cols],
    "Example values": [
        ", ".join(linkedin[c].dropna().astype(str).unique()[:4])
        for c in meta_cols
    ],
})
display(meta)


### 6.5 Repetition and template detection


In [ ]:
li_div = (
    linkedin.dropna(subset=["title", "description"])
    .groupby("title")["description"]
    .nunique()
    .sort_values(ascending=False)
    .reset_index(name="distinct_descriptions")
)

print("Distinct descriptions per title — summary:")
display(li_div.describe().round(0))
print("\nTop 15 titles by description variety:")
display(li_div.head(15))


## 7. Comparative analysis

Side-by-side metrics that directly compare Djinni and LinkedIn
on dimensions relevant to fine-tuning.


### 7.1 Target text quality

In [ ]:
target_cols = {"Djinni": "Long Description", "LinkedIn": "description"}

rows_tq = []
for name, col in target_cols.items():
    t = datasets[name][col].dropna().astype(str)
    rows_tq.append({
        "Dataset":        name,
        "Target column":  col,
        "Non-null":       len(t),
        "Unique":         t.nunique(),
        "Duplicate ratio": round(1 - t.nunique() / len(t), 4),
        "Avg length":     round(t.str.len().mean(), 0),
        "Median length":  round(t.str.len().median(), 0),
        "Min length":     int(t.str.len().min()),
        "Max length":     int(t.str.len().max()),
    })

display(pd.DataFrame(rows_tq))


### 7.2 Lexical diversity

Lexical diversity (unique tokens / total tokens on a sample) measures how varied
the vocabulary is. Higher diversity → the model is exposed to richer language.


In [ ]:
import re

def lexical_diversity(series, sample_size=5000):
    text = " ".join(
        series.dropna().astype(str).head(sample_size).str.lower().tolist()
    )
    words = re.findall(r"\b[a-zA-Z][a-zA-Z0-9+#.-]{2,}\b", text)
    if not words:
        return 0, 0, 0.0
    return len(words), len(set(words)), round(len(set(words)) / len(words), 4)

rows_ld = []
for name, col in target_cols.items():
    total, unique, ratio = lexical_diversity(datasets[name][col])
    rows_ld.append({
        "Dataset": name, "Column": col,
        "Total tokens (sample)": total,
        "Unique tokens": unique,
        "Lexical diversity": ratio,
    })

display(pd.DataFrame(rows_ld))


### 7.3 Most common terms

A qualitative view of the vocabulary each dataset uses most.


In [ ]:
from collections import Counter

STOPWORDS = {
    "the", "and", "for", "with", "you", "are", "our", "will", "this", "that",
    "from", "have", "your", "job", "work", "team", "experience", "skills",
    "candidate", "company", "role", "responsibilities", "requirements"
}

def top_terms(series, n=20):
    text = " ".join(series.dropna().astype(str).head(10_000).str.lower())
    words = re.findall(r"\b[a-zA-Z][a-zA-Z0-9+#.-]{2,}\b", text)
    words = [w for w in words if w not in STOPWORDS]
    return pd.DataFrame(Counter(words).most_common(n), columns=["term", "count"])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (name, col) in zip(axes, target_cols.items()):
    terms = top_terms(datasets[name][col])
    ax.barh(terms["term"][::-1], terms["count"][::-1], color="steelblue")
    ax.set_title(f"{name} — top 20 terms")
    ax.set_xlabel("occurrences")
plt.tight_layout()
plt.show()


## 8. Prompt-readiness and recommendations

### 8.1 Prompt-readiness score

A row is **prompt-ready** when the target description is non-null and ≥ 200 chars,
and at least one structured input feature is non-null.
This gives the realistic ceiling of usable `(prompt → completion)` pairs.


In [ ]:
MIN_LEN = 200

rows_pr = []

# Djinni
has_desc = djinni["Long Description"].notna() & (djinni["Long Description"].str.len() >= MIN_LEN)
has_feat = djinni[["Primary Keyword", "Exp Years"]].notna().any(axis=1)
ready = has_desc & has_feat
rows_pr.append({
    "Dataset": "Djinni",
    "Total rows": len(djinni),
    "Prompt-ready rows": int(ready.sum()),
    "Ready %": round(ready.mean() * 100, 1),
    "Dropped — short/null desc": int((~has_desc).sum()),
    "Dropped — no feature": int((has_desc & ~has_feat).sum()),
})

# LinkedIn
has_desc_li = linkedin["description"].notna() & (linkedin["description"].str.len() >= MIN_LEN)
has_feat_li = linkedin[["title", "formatted_experience_level"]].notna().any(axis=1)
ready_li = has_desc_li & has_feat_li
rows_pr.append({
    "Dataset": "LinkedIn",
    "Total rows": len(linkedin),
    "Prompt-ready rows": int(ready_li.sum()),
    "Ready %": round(ready_li.mean() * 100, 1),
    "Dropped — short/null desc": int((~has_desc_li).sum()),
    "Dropped — no feature": int((has_desc_li & ~has_feat_li).sum()),
})

display(pd.DataFrame(rows_pr))


### 8.2 Recommendations

| # | Finding | Implication |
|---|---------|-------------|
| 1 | Djinni is IT-only; LinkedIn spans all industries. | Use Djinni for train/val; LinkedIn as out-of-domain test only. |
| 2 | Description length is right-skewed (long tail > 5k chars) in both datasets. | Apply a max-token cap (~512–1024) during tokenisation, or truncate long descriptions. |
| 3 | `Exp Years` and `English Level` together cover > 95 % of Djinni rows and co-vary (see §5.4). | Include both as prompt fields; the cross-tab shows they add complementary signal. |
| 4 | `skills_desc` in LinkedIn has significant missingness (see §6.1). | Do not rely on `skills_desc` as a required field for LinkedIn evaluation. |
| 5 | Description length correlates with seniority in Djinni (§5.5). | Providing `Exp Years` in prompts helps the model calibrate output verbosity. |
| 6 | LinkedIn is dominated by Engineering/Dev and Data/AI (~50 % combined, §6.3). | The out-of-domain test is skewed; report per-family generalisation metrics separately. |


## 9. Skill → Role mapping analysis

This section is directly aligned with the project objective: **informal business requirements → inferred role → structured job description**.

Instead of only checking dataset size or missing values, we verify whether technical skills and domain terms are meaningfully associated with job roles. If this relationship exists, the dataset is suitable for teaching a language model to infer roles from loose recruiter or business-area descriptions.

**Implication for fine-tuning:** if skills such as `Figma`, `UX`, `Kubernetes`, `SQL`, `Azure`, or `CI/CD` consistently appear near specific job families, then the model can learn useful semantic mappings rather than memorising isolated templates.


In [ ]:
import re
from collections import Counter

SKILL_PATTERNS = {
    "Figma":                r"\bfigma\b",
    "UX/UI":                r"\bux\b|\bui\b|user experience|user interface",
    "Design Systems":       r"design system",
    "SQL":                  r"\bsql\b|postgres|mysql|sql server|oracle",
    "Azure":                r"\bazure\b",
    "AWS":                  r"\baws\b|amazon web services",
    "GCP":                  r"\bgcp\b|google cloud",
    "Kubernetes":           r"kubernetes|\bk8s\b",
    "Docker":               r"\bdocker\b|container",
    "CI/CD":                r"ci/cd|continuous integration|continuous delivery|pipeline",
    "Python":               r"\bpython\b",
    "Java":                 r"\bjava\b",
    "JavaScript":           r"javascript|\bjs\b|typescript|\bts\b",
    "React":                r"\breact\b|reactjs",
    "Scrum/Agile":          r"\bscrum\b|\bagile\b|kanban",
    "Product Ownership":    r"product owner|backlog|roadmap|user stories",
    "Cybersecurity":        r"security|cyber|siem|iam|vulnerabilit|soc",
    "Architecture":         r"architect|architecture|solution design",
    "Stakeholder Mgmt":     r"stakeholder|business team|cross-functional|multi[- ]team",
}

def extract_skills(text):
    t = str(text).lower()
    return [skill for skill, pat in SKILL_PATTERNS.items()
            if re.search(pat, t, flags=re.IGNORECASE)]

# ── Build skill_df with explode instead of iterrows ──────────────────────────
base = djinni.dropna(subset=["Long Description", "Primary Keyword"]).copy()
base["extracted_skills"] = base["Long Description"].apply(extract_skills)

# explode: one row per (description, skill) — no Python loop needed
skill_df = (
    base[["Primary Keyword", "extracted_skills"]]
    .explode("extracted_skills")
    .dropna(subset=["extracted_skills"])
    .rename(columns={"Primary Keyword": "role_or_domain", "extracted_skills": "skill"})
    .reset_index(drop=True)
)

n_with_skill = (base["extracted_skills"].str.len() > 0).sum()
print(f"Rows with ≥1 extracted skill : {n_with_skill:,}  ({n_with_skill/len(base):.1%})")
print(f"Total skill mentions         : {len(skill_df):,}")

if skill_df.empty:
    print("No skills extracted — check SKILL_PATTERNS against the actual vocabulary.")
else:
    top_skill_counts = skill_df["skill"].value_counts().reset_index()
    top_skill_counts.columns = ["skill", "mentions"]
    display(top_skill_counts)

    ax = top_skill_counts.head(20).iloc[::-1].plot.barh(
        x="skill", y="mentions", legend=False, figsize=(8, 5),
        title="Djinni — extracted skill mentions")
    ax.set_xlabel("mentions")
    plt.tight_layout()
    plt.show()



### 9.1 Most associated role/domain per skill

The table below answers a core project question: *when a business user mentions a skill, which role/domain is the dataset most likely to associate with it?*

This supports the role-inference part of the project. For example, if `Figma` and `UX/UI` concentrate around design-oriented roles, or `Kubernetes` and `CI/CD` around DevOps/cloud roles, the model has useful patterns to learn.


In [ ]:
if not skill_df.empty:
    # Top-3 roles per skill
    skill_role = (
        skill_df.groupby(["skill", "role_or_domain"])
        .size()
        .reset_index(name="count")
        .sort_values(["skill", "count"], ascending=[True, False])
    )
    most_associated = skill_role.groupby("skill").head(3)
    display(most_associated)

    # Heatmap: use size() instead of count() on a text column
    top_skills = skill_df["skill"].value_counts().head(12).index
    top_roles  = skill_df["role_or_domain"].value_counts().head(15).index

    pivot = (
        skill_df[
            skill_df["skill"].isin(top_skills) &
            skill_df["role_or_domain"].isin(top_roles)
        ]
        .groupby(["skill", "role_or_domain"])
        .size()
        .unstack(fill_value=0)
    )
    display(pivot)

    fig, ax = plt.subplots(figsize=(12, 6))
    im = ax.imshow(pivot.values, aspect="auto", cmap="Blues")
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns, rotation=45, ha="right")
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index)
    ax.set_title("Skill ↔ role/domain co-occurrence heatmap")
    fig.colorbar(im, ax=ax, label="co-occurrences")
    plt.tight_layout()
    plt.show()


## 10. Tokenisation analysis for model training

Character length is useful, but LLM training is constrained by **tokens**, not characters. This section estimates token counts using the intended model tokenizer.

**Implication for fine-tuning:** this analysis justifies `max_length`, truncation strategy, batch size and compute budget. Long tails in token length may require truncation or filtering to avoid wasting training budget on extremely long postings.


In [ ]:
MODEL_NAME = "HuggingFaceTB/SmolLM2-360M"   # base model used for MP1
SAMPLE_FOR_TOKEN_ANALYSIS = 5000

try:
    from transformers import AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    print(f"Loaded tokenizer: {MODEL_NAME}\n")

    def count_tokens_batch(texts, batch_size=128):
        counts = []
        for i in range(0, len(texts), batch_size):
            batch = [str(t) for t in texts[i:i + batch_size]]
            enc = tokenizer(batch, add_special_tokens=True, truncation=False)
            counts.extend(len(ids) for ids in enc["input_ids"])
        return counts

    token_rows = []
    for dataset_name, col in target_cols.items():
        non_null = datasets[dataset_name][col].dropna().astype(str)
        sample   = non_null.sample(min(SAMPLE_FOR_TOKEN_ANALYSIS, len(non_null)), random_state=42)
        counts   = pd.Series(count_tokens_batch(sample.tolist()), name="tokens")

        token_rows.append({
            "Dataset":        dataset_name,
            "Sample size":    len(sample),
            "Mean tokens":    round(counts.mean(), 1),
            "Median tokens":  round(counts.median(), 1),
            "P90":            int(counts.quantile(0.90)),
            "P95":            int(counts.quantile(0.95)),
            "P99":            int(counts.quantile(0.99)),
            "Max":            int(counts.max()),
            "% > 512":        round((counts > 512).mean() * 100, 1),
            "% > 1024":       round((counts > 1024).mean() * 100, 1),
        })

        ax = counts.clip(upper=2048).plot.hist(
            bins=60, figsize=(8, 3),
            title=f"{dataset_name} — token length distribution (clipped at 2 048)")
        ax.set_xlabel("tokens")
        plt.tight_layout()
        plt.show()

    display(pd.DataFrame(token_rows))

except Exception as e:
    print("Tokenizer analysis skipped — install transformers and verify model access.")
    print("Error:", repr(e))


### 10.1 Technical-term fragmentation

A practical concern for IT job descriptions: domain-specific terms like
`Kubernetes`, `CI/CD`, or `TypeScript` may be split into many sub-word tokens
by a general-purpose tokenizer. Heavy fragmentation means:

1. the model sees the concept spread across more positions → harder to learn,
2. token budget is consumed faster than the character count suggests.

The cell below tokenises a sample of domain-critical terms and reports how many
tokens each one produces.


In [ ]:
TECHNICAL_TERMS = [
    "Kubernetes", "CI/CD", "TypeScript", "PostgreSQL", "Dockerfile",
    "microservices", "DevOps", "GitLab", "Terraform", "React.js",
    "FastAPI", "PyTorch", "LangChain", "OpenTelemetry", "GraphQL",
    "RabbitMQ", "ElasticSearch", "Helm", "ArgoCD", "Prometheus"
]

try:
    rows_frag = []
    for term in TECHNICAL_TERMS:
        ids     = tokenizer.encode(term, add_special_tokens=False)
        tokens  = tokenizer.convert_ids_to_tokens(ids)
        rows_frag.append({
            "term":        term,
            "n_tokens":    len(ids),
            "sub-tokens":  " | ".join(tokens),
        })

    frag_df = pd.DataFrame(rows_frag).sort_values("n_tokens", ascending=False)
    display(frag_df)

    ax = frag_df.sort_values("n_tokens").plot.barh(
        x="term", y="n_tokens", legend=False, figsize=(7, 5),
        title=f"Token fragmentation of technical terms ({MODEL_NAME})")
    ax.set_xlabel("number of sub-word tokens")
    ax.axvline(1, color="red", linestyle="--", linewidth=0.8, label="ideal (1 token)")
    ax.legend()
    plt.tight_layout()
    plt.show()

except NameError:
    print("Tokenizer not loaded — run the cell above first.")

## 11. Semantic embedding analysis

This is one of the most important LLM-oriented analyses in the notebook. It checks whether job descriptions form meaningful semantic clusters.

If UX/design, DevOps/cloud, data/AI and security postings naturally cluster, this supports the assumption that a model can learn role-relevant patterns from the dataset.

The code first tries to use `sentence-transformers`. If unavailable, it falls back to TF-IDF + dimensionality reduction so that the notebook remains runnable in lighter environments.


In [ ]:
import numpy as np
from sklearn.decomposition import TruncatedSVD, PCA
from sklearn.feature_extraction.text import TfidfVectorizer

SAMPLE_FOR_EMBEDDINGS = 1500
RANDOM_STATE = 42

sem_df = (
    djinni.dropna(subset=["Long Description", "Primary Keyword"])
    .sample(min(SAMPLE_FOR_EMBEDDINGS, len(djinni)), random_state=RANDOM_STATE)
    .reset_index(drop=True)
)
texts_sem  = sem_df["Long Description"].astype(str).tolist()
labels_sem = sem_df["Primary Keyword"].astype(str).tolist()

try:
    from sentence_transformers import SentenceTransformer
    emb_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
    embeddings = emb_model.encode(texts_sem, show_progress_bar=True,
                                   normalize_embeddings=True)
    embedding_source = "sentence-transformers/all-MiniLM-L6-v2"
except Exception as e:
    print("sentence-transformers unavailable — falling back to TF-IDF + SVD.")
    print("Note: clusters in this mode reflect shared vocabulary, not semantics.")
    print("Error:", repr(e))
    vec  = TfidfVectorizer(max_features=5000, stop_words="english", min_df=3)
    X    = vec.fit_transform(texts_sem)
    svd  = TruncatedSVD(n_components=100, random_state=RANDOM_STATE)
    embeddings = svd.fit_transform(X)
    embedding_source = "TF-IDF + TruncatedSVD (vocabulary overlap, not semantics)"

print("Embedding source:", embedding_source)
print("Matrix shape    :", embeddings.shape)



### 11.1 2D projection of semantic clusters

This visualisation is not used as a model metric. It is an exploratory diagnostic to understand whether the dataset contains separable role/domain families.


In [ ]:
top_label_count = 12
top_labels  = pd.Series(labels_sem).value_counts().head(top_label_count).index
plot_labels = [x if x in top_labels else "Other" for x in labels_sem]

try:
    import umap
    reducer = umap.UMAP(n_neighbors=20, min_dist=0.1, random_state=RANDOM_STATE)
    coords  = reducer.fit_transform(embeddings)
    method  = "UMAP"
except Exception as e:
    print("UMAP unavailable — falling back to PCA.")
    coords = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(embeddings)
    method = "PCA"

plot_df = pd.DataFrame({"x": coords[:, 0], "y": coords[:, 1], "label": plot_labels})

fig, ax = plt.subplots(figsize=(10, 7))
for label, grp in plot_df.groupby("label"):
    ax.scatter(grp["x"], grp["y"], s=12, alpha=0.65, label=label)
ax.set_title(f"Djinni semantic clusters — {method} ({embedding_source})")
ax.set_xlabel("dim 1");  ax.set_ylabel("dim 2")
ax.legend(markerscale=2, fontsize=8, bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()



### 11.2 Nearest-neighbour inspection

Cluster plots can be subjective. Nearest-neighbour examples provide a more concrete check: for a selected posting, do the closest descriptions refer to similar roles/domains?


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Normalise once
norms  = np.linalg.norm(embeddings, axis=1, keepdims=True)
E_norm = embeddings / np.maximum(norms, 1e-12)

def show_neighbours(query_idx, n=5):
    sims   = cosine_similarity(E_norm[query_idx:query_idx+1], E_norm).ravel()
    nbrs   = sims.argsort()[::-1][1:n+1]
    print(f"{'='*70}")
    print(f"QUERY  #{query_idx}  |  role/domain: {labels_sem[query_idx]}")
    print(texts_sem[query_idx][:500], "...\n")
    for rank, idx in enumerate(nbrs, 1):
        print(f"  #{rank} | sim={sims[idx]:.3f} | role/domain: {labels_sem[idx]}")
        print(" ", texts_sem[idx][:300], "...\n")

# Pick one representative index for each target domain.
# The search below selects the first occurrence of each keyword in the sample.
target_domains = ["DevOps", "Data Science", "UI/UX Designer"]
for domain in target_domains:
    hits = [i for i, l in enumerate(labels_sem) if domain.lower() in l.lower()]
    if hits:
        show_neighbours(hits[0])
    else:
        print(f"Domain '{domain}' not found in sample — try adjusting the list.")



## 12. Building the continued-pretraining corpus

The exploration above (§4–§11) settled the design; this section **applies** it,
turning the raw datasets into the fixed corpus the model trains on. Continued
pretraining needs only **raw text** — no labels, no queries — so each split is
just one `{"text": ...}` record per line.

Build parameters (following the §8 recommendations):

| Parameter | Value | Rationale |
|---|---|---|
| Length bounds | 200–8000 chars | drop stub postings and outlier walls of text (§5.3, §6.2) |
| LinkedIn dedup | yes | LinkedIn reposts share identical text (§6.5) |
| Djinni split | 12 000 train / 1 000 val / 1 000 test | disjoint, in-domain |
| LinkedIn split | 1 000 `ood_test` | out-of-domain, never trained on |
| Seed | 42 | reproducible shuffle |

This is the same logic as the standalone `src/prepare_corpus.py`; the cell below
runs it directly, reusing the dataframes already loaded in §4.

In [ ]:
import json
import random

MP1_CORPUS = DATA / "mp1"
MP1_CORPUS.mkdir(parents=True, exist_ok=True)

SEED = 42
MIN_CHARS, MAX_CHARS = 200, 8000             # drop stubs and outlier walls of text
N_TRAIN, N_VAL, N_TEST, N_OOD = 12_000, 1_000, 1_000, 1_000


def clean(series):
    """Drop nulls, strip whitespace, keep only descriptions within length bounds."""
    s = series.dropna().astype(str).str.strip()
    return s[s.str.len().between(MIN_CHARS, MAX_CHARS)]


def write_split(name, texts):
    """Write one {"text": ...} record per line; return a summary row."""
    path = MP1_CORPUS / f"{name}.jsonl"
    with open(path, "w", encoding="utf-8") as f:
        for t in texts:
            f.write(json.dumps({"text": t}, ensure_ascii=False) + "\n")
    chars = sum(len(t) for t in texts)
    return {"split": name, "docs": len(texts),
            "MB": round(path.stat().st_size / 1e6, 1),
            "~tokens (est.)": chars // 4}


rng = random.Random(SEED)

# Djinni -> in-domain train / val / test (disjoint slices of one shuffled pool)
dj_texts = clean(djinni["Long Description"]).tolist()
assert len(dj_texts) >= N_TRAIN + N_VAL + N_TEST, "not enough clean Djinni docs"
rng.shuffle(dj_texts)
train = dj_texts[:N_TRAIN]
val   = dj_texts[N_TRAIN:N_TRAIN + N_VAL]
test  = dj_texts[N_TRAIN + N_VAL:N_TRAIN + N_VAL + N_TEST]

# LinkedIn -> out-of-domain test (deduplicated; never used for training)
lk_texts = clean(linkedin["description"]).drop_duplicates().tolist()
rng.shuffle(lk_texts)
ood = lk_texts[:N_OOD]

summary = [write_split("train", train), write_split("val", val),
           write_split("test", test), write_split("ood_test", ood)]
print(f"MP1 corpus written to {MP1_CORPUS}/  (seed={SEED})")
pd.DataFrame(summary).set_index("split")

The corpus is now fixed on disk at `data/processed/mp1/`. The
continued-pretraining runs in §13 train on `train` and validate on `val`; §14
evaluates perplexity on the held-out `test` (in-domain) and `ood_test`
(out-of-domain).

## 13. Continued pretraining — full fine-tuning vs LoRA

MP1 continues the **pretraining** of the base `SmolLM2-360M` on the raw
job-description corpus in `data/processed/mp1/` — plain next-token prediction,
no labels and no queries.

**Optimization experiment (the required knob):** full fine-tuning vs. LoRA,
with everything else held constant (effective batch size, epochs, schedule, data).

> The heavy runs are also packaged as a standalone script,
> `src/train.py --mode full|lora` — that is how they were executed here (in the
> background, each in its own crash-isolated process). The cells below are the
> same logic, runnable directly in the notebook.

In [ ]:
import itertools, json, math, time

import torch
from datasets import load_dataset
from transformers import (AutoModelForCausalLM, AutoTokenizer,
                          DataCollatorForLanguageModeling, Trainer,
                          TrainingArguments, set_seed)
from peft import LoraConfig, get_peft_model

MODEL = "HuggingFaceTB/SmolLM2-360M"            # base model (NOT -Instruct)
RUN = "mp1-360m"                                # this run's output-folder tag
CORPUS = PROJECT_ROOT / "data" / "processed" / "mp1"
OUTPUTS = PROJECT_ROOT / "outputs"
RUN_DIR = OUTPUTS / RUN                         # outputs/mp1-360m/  (full/ lora/ eval.json)
BLOCK_SIZE = 1024
SEED = 42

# --- held constant across both modes ---
EPOCHS = 3
PER_DEVICE_BATCH = 4
GRAD_ACCUM = 4                                  # effective batch = 16
WARMUP_RATIO = 0.03
WEIGHT_DECAY = 0.01

# --- the experiment knob: full fine-tuning vs LoRA ---
MODE_CFG = {
    "full": {"learning_rate": 5e-5},
    "lora": {"learning_rate": 2e-4, "r": 16, "alpha": 32, "dropout": 0.05,
             "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj",
                                "gate_proj", "up_proj", "down_proj"]},
}
print("config ready — base model:", MODEL)

In [ ]:
def tokenize_and_chunk(jsonl_path, tokenizer, block_size=BLOCK_SIZE):
    """Tokenize raw text, mark doc boundaries with EOS, pack into fixed blocks."""
    ds = load_dataset("json", data_files=str(jsonl_path), split="train")

    def tok(batch):
        out = tokenizer(batch["text"])
        for ids in out["input_ids"]:
            ids.append(tokenizer.eos_token_id)
        return {"input_ids": out["input_ids"]}

    ds = ds.map(tok, batched=True, remove_columns=ds.column_names)

    def group(batch):
        ids = list(itertools.chain.from_iterable(batch["input_ids"]))
        n = (len(ids) // block_size) * block_size
        blocks = [ids[i:i + block_size] for i in range(0, n, block_size)]
        return {"input_ids": blocks,
                "attention_mask": [[1] * block_size for _ in blocks]}

    return ds.map(group, batched=True)

In [ ]:
def run_training(mode):
    """Continued pretraining in `mode` ('full' or 'lora'); returns a summary dict."""
    cfg = MODE_CFG[mode]
    set_seed(SEED)
    out_dir = RUN_DIR / mode

    tokenizer = AutoTokenizer.from_pretrained(MODEL)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(MODEL)

    if mode == "lora":
        model = get_peft_model(model, LoraConfig(
            r=cfg["r"], lora_alpha=cfg["alpha"], lora_dropout=cfg["dropout"],
            target_modules=cfg["target_modules"], task_type="CAUSAL_LM"))
        model.print_trainable_parameters()
    else:
        print("full fine-tuning — all parameters trainable")

    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())

    train_ds = tokenize_and_chunk(CORPUS / "train.jsonl", tokenizer)
    val_ds = tokenize_and_chunk(CORPUS / "val.jsonl", tokenizer)
    print(f"train blocks: {len(train_ds):,} | val blocks: {len(val_ds):,}")

    args = TrainingArguments(
        output_dir=str(out_dir),
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=PER_DEVICE_BATCH,
        per_device_eval_batch_size=PER_DEVICE_BATCH,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=cfg["learning_rate"],
        warmup_ratio=WARMUP_RATIO,
        weight_decay=WEIGHT_DECAY,
        bf16=True,
        eval_strategy="steps", eval_steps=50, logging_steps=10,
        save_strategy="no", seed=SEED, report_to=[],
    )
    trainer = Trainer(
        model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
        data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
        processing_class=tokenizer,
    )

    t0 = time.time()
    trainer.train()
    minutes = (time.time() - t0) / 60

    out_dir.mkdir(parents=True, exist_ok=True)
    trainer.save_model(str(out_dir))
    tokenizer.save_pretrained(str(out_dir))

    final = trainer.evaluate()
    summary = {"mode": mode, "learning_rate": cfg["learning_rate"],
               "trainable_params": trainable_params, "total_params": total_params,
               "minutes": round(minutes, 1),
               "final_val_loss": round(final["eval_loss"], 4),
               "val_perplexity": round(math.exp(final["eval_loss"]), 2)}
    (out_dir / "log_history.json").write_text(
        json.dumps(trainer.state.log_history, indent=1))
    (out_dir / "summary.json").write_text(json.dumps(summary, indent=1))
    print("done:", summary)
    return summary

**Run A — full fine-tuning.** Updates *every* parameter of the model. The exact
trainable-parameter count and wall-clock time are reported by the cell's output
and recorded in this run's `full/summary.json`.

In [ ]:
summary_full = run_training("full")

**Run B — LoRA.** Trains only the small low-rank adapter matrices, leaving the
base weights frozen — `print_trainable_parameters()` reports how small that
fraction is.

Note the modest `PER_DEVICE_BATCH` (4, with grad-accum 4): LoRA hit a CUDA OOM at
micro-batch 16 in this `transformers 5.9 / peft 0.19` stack, where full
fine-tuning did not. The **effective** batch size (16) is held equal so the
comparison stays controlled — see the report's *difficulties* section.

In [ ]:
summary_lora = run_training("lora")

## 14. Results

This section evaluates the three checkpoints — untrained **base**, **full
fine-tuning** and **LoRA** — entirely within the notebook: it plots the training
loss curves, computes test-set perplexity and sample generations, summarises, and
finally compares this 360M run against the 135M run. **Every number below is
computed from the run artefacts in `outputs/` — nothing is hardcoded**, so
re-running these cells after a fresh training run refreshes the whole section.

Evaluation uses two held-out test sets: **in-domain** (Djinni IT postings the
model never saw) and **out-of-domain / OOD** (LinkedIn, cross-industry) — the
latter probing whether the domain adaptation transfers beyond IT. The same
evaluation is also packaged as `src/evaluate_mp1.py`.

In [ ]:
import json



def load_curves(mode):
    """Return (train_steps, train_loss), (eval_steps, eval_loss) for a run."""
    hist = json.loads((RUN_DIR / mode / "log_history.json").read_text())
    train = [(e["step"], e["loss"]) for e in hist if "loss" in e]
    evals = [(e["step"], e["eval_loss"]) for e in hist if "eval_loss" in e]
    return train, evals


fig, ax = plt.subplots(figsize=(8, 5))
for mode, color in [("full", "tab:blue"), ("lora", "tab:orange")]:
    train, evals = load_curves(mode)
    ts, tl = zip(*train)
    es, el = zip(*evals)
    ax.plot(ts, tl, color=color, alpha=0.35, linewidth=1, label=f"{mode} — train")
    ax.plot(es, el, color=color, marker="o", markersize=4, label=f"{mode} — eval")

ax.set_xlabel("training step")
ax.set_ylabel("cross-entropy loss")
ax.set_title("MP1 continued pretraining — loss curves (3 epochs, effective batch 16)")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

Both curves should descend smoothly across the three epochs with no divergence.
Their relative standing — and whether LoRA, training only a small fraction of the
parameters, keeps up with full fine-tuning — is quantified next as test-set
perplexity.

**Evaluating the three checkpoints.** The cell below scores `base`, `full` and
`lora` on held-out **perplexity** — in-domain (Djinni `test`) and out-of-domain
(LinkedIn `ood_test`) — and produces greedy sample generations from each. It
writes this run's `eval.json` (under `outputs/mp1-360m/`); every table and
generation below reads from it. Same logic as `src/evaluate_mp1.py`, inlined so
the notebook runs end to end.

In [ ]:
import json
import math

import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

EVAL_BLOCK = 1024
DEVICE = "cuda"
DTYPE = torch.bfloat16
EVAL_SETS = {"in_domain": CORPUS / "test.jsonl", "ood": CORPUS / "ood_test.jsonl"}
EVAL_PROMPTS = ["We are looking for a", "The ideal candidate will", "Responsibilities:\n-"]


def eval_base_id():
    """Base model id, read from this run's LoRA adapter config — never hardcoded."""
    cfg = json.loads((RUN_DIR / "lora" / "adapter_config.json").read_text())
    return cfg["base_model_name_or_path"]


def make_eval_blocks(jsonl_path, tokenizer):
    """Tokenize a jsonl corpus and pack into fixed-size blocks for perplexity."""
    ds = load_dataset("json", data_files=str(jsonl_path), split="train")
    ids = []
    for text in ds["text"]:
        ids.extend(tokenizer(text)["input_ids"])
        ids.append(tokenizer.eos_token_id)
    n = (len(ids) // EVAL_BLOCK) * EVAL_BLOCK
    return [ids[i:i + EVAL_BLOCK] for i in range(0, n, EVAL_BLOCK)]


@torch.no_grad()
def block_perplexity(model, blocks):
    """Token-weighted perplexity over a list of equal-length blocks."""
    tot_loss, tot_tok = 0.0, 0
    for i in range(0, len(blocks), 8):
        batch = torch.tensor(blocks[i:i + 8], device=DEVICE)
        loss = model(batch, labels=batch).loss
        n = batch.shape[0] * (batch.shape[1] - 1)
        tot_loss += loss.item() * n
        tot_tok += n
    return math.exp(tot_loss / tot_tok)


def load_eval_model(name):
    """Load 'base' | 'full' | 'lora' for this run onto the GPU in eval mode."""
    if name == "lora":
        from peft import PeftModel
        m = AutoModelForCausalLM.from_pretrained(eval_base_id())
        m = PeftModel.from_pretrained(m, str(RUN_DIR / "lora"))
    elif name == "full":
        m = AutoModelForCausalLM.from_pretrained(str(RUN_DIR / "full"))
    else:  # base
        m = AutoModelForCausalLM.from_pretrained(eval_base_id())
    return m.to(DEVICE, dtype=DTYPE).eval()


eval_tok = AutoTokenizer.from_pretrained(eval_base_id())
if eval_tok.pad_token is None:
    eval_tok.pad_token = eval_tok.eos_token
eval_blocks = {name: make_eval_blocks(path, eval_tok)
               for name, path in EVAL_SETS.items()}

eval_results = {"perplexity": {}, "generations": {}}
for name in ("base", "full", "lora"):
    print(f"evaluating {name} ...", flush=True)
    model = load_eval_model(name)
    eval_results["perplexity"][name] = {
        s: round(block_perplexity(model, b), 2) for s, b in eval_blocks.items()}
    gens = []
    for prompt in EVAL_PROMPTS:
        enc = eval_tok(prompt, return_tensors="pt").to(DEVICE)
        out = model.generate(**enc, max_new_tokens=80, do_sample=False,
                             pad_token_id=eval_tok.eos_token_id)
        gens.append({"prompt": prompt,
                     "output": eval_tok.decode(out[0], skip_special_tokens=True)})
    eval_results["generations"][name] = gens
    del model
    torch.cuda.empty_cache()

(RUN_DIR / "eval.json").write_text(
    json.dumps(eval_results, indent=2, ensure_ascii=False))
print(f"evaluation written -> {RUN_DIR / 'eval.json'}")

In [ ]:
evald = json.loads((RUN_DIR / "eval.json").read_text())
ppl = evald["perplexity"]
base = ppl["base"]

rows = []
for model in ["base", "full", "lora"]:
    p = ppl[model]
    rows.append({
        "model": {"base": "base SmolLM2-360M",
                  "full": "full fine-tuning",
                  "lora": "LoRA r=16"}[model],
        "in-domain ppl": p["in_domain"],
        "OOD ppl": p["ood"],
        "in-domain vs base": f'{(p["in_domain"] / base["in_domain"] - 1) * 100:+.0f}%',
        "OOD vs base": f'{(p["ood"] / base["ood"] - 1) * 100:+.0f}%',
    })

ppl_table = pd.DataFrame(rows).set_index("model")
print("Test-set perplexity (lower = better)\n")
ppl_table

**Reading the table.** The Δ-columns show each run's perplexity change against
the untrained base. Two things to look for:

- **In-domain adaptation** — how far in-domain perplexity drops. A large drop
  means the model has absorbed job-posting style and vocabulary.
- **Generalisation** — how much OOD perplexity moves. If it barely changes while
  in-domain drops sharply, the adaptation is **domain-specific**: it does not
  transfer to cross-industry postings. That contrast is the generalisation
  question the experiment was designed to answer.

**Honest caveat — full FT vs LoRA.** The two runs are controlled on effective
batch size, epochs, schedule shape and data, but **not** the learning rate
(LoRA `2e-4` vs full FT `5e-5`; each keeps its conventional working range). Any
in-domain gap between the two methods therefore carries that LR confound — worth
stating plainly rather than smoothing over.

In [ ]:
gens = evald["generations"]
prompts = [g["prompt"] for g in gens["base"]]

for i, prompt in enumerate(prompts):
    print("=" * 80)
    print(f"PROMPT: {prompt!r}")
    print("=" * 80)
    for model in ["base", "full", "lora"]:
        print(f"\n--- {model} ---")
        print(gens[model][i]["output"])
    print()

**Generations (greedy decoding).** Compare the three blocks above. The untrained
`base` model typically loops and degenerates; after continued pretraining both
trained runs produce recognisable job-posting prose — responsibilities,
"We offer", experience requirements. Repetition under greedy decoding still
persists, expected for a model of this size — MP2's SFT + alignment stage targets
exactly this output quality.

In [ ]:
# MP1 summary — every value computed from this run's artefacts (nothing hardcoded).
ppl = json.loads((RUN_DIR / "eval.json").read_text())["perplexity"]
base_ppl = ppl["base"]


def vs_base(value, ref):
    return f"{value} ({(value / ref - 1) * 100:+.0f}%)"


summary_rows = {}
for mode, label in [("full", "full fine-tuning"), ("lora", "LoRA r=16")]:
    s = json.loads((RUN_DIR / mode / "summary.json").read_text())
    trn, tot = s["trainable_params"], s["total_params"]
    summary_rows[label] = {
        "trainable params": f"{trn / 1e6:.1f}M ({trn / tot * 100:.1f}%)",
        "learning rate": s["learning_rate"],
        "wall-clock (min)": s["minutes"],
        "in-domain test ppl": vs_base(ppl[mode]["in_domain"], base_ppl["in_domain"]),
        "OOD test ppl": vs_base(ppl[mode]["ood"], base_ppl["ood"]),
    }

best = min(["full", "lora"], key=lambda m: ppl[m]["in_domain"])
print(f"base model  —  in-domain ppl {base_ppl['in_domain']}, OOD ppl {base_ppl['ood']}")
print(f"best in-domain fit: {best}  ->  checkpoint carried into MP2")
pd.DataFrame(summary_rows)

### Model-size comparison — 135M vs 360M

MP1 was run at two model sizes: `SmolLM2-135M` (notebook `atlm_mp1_v2.ipynb`,
run tag `mp1-135m`) and `SmolLM2-360M` (this notebook, run tag `mp1-360m`). Each
run's results live in its own folder under `outputs/`, so they never overwrite
each other. The cell below reads both `eval.json` files and puts their test
perplexity side by side — the `135M` column appears once `atlm_mp1_v2.ipynb`
has been run.

In [ ]:
# 135M vs 360M — reads each run's eval.json (whichever exist on disk).
COMPARE = {"135M (v2)": "mp1-135m", "360M (v3)": "mp1-360m"}

ppl_by_run = {}
for label, run in COMPARE.items():
    path = OUTPUTS / run / "eval.json"
    if path.exists():
        ppl_by_run[label] = json.loads(path.read_text())["perplexity"]
    else:
        print(f"{label}: outputs/{run}/eval.json not found — run that notebook first.")

if ppl_by_run:
    rows = []
    for variant in ("base", "full", "lora"):
        row = {"variant": variant}
        for label, ppl in ppl_by_run.items():
            row[f"{label} · in-domain"] = ppl[variant]["in_domain"]
            row[f"{label} · OOD"] = ppl[variant]["ood"]
        rows.append(row)
    cmp_table = pd.DataFrame(rows).set_index("variant")
    print("MP1 model-size comparison — test perplexity (lower = better)\n")
    display(cmp_table)
else:
    print("No runs available to compare yet.")

### Takeaway

Continued pretraining adapts the model to the job-postings domain — in-domain
perplexity drops for both training methods, while OOD perplexity (LinkedIn,
cross-industry) moves far less. The adaptation is **real but domain-specific**:
an IT-trained model does not generalise to other industries.

The full-fine-tuning vs LoRA comparison is controlled on everything except the
learning rate, so any in-domain gap between them carries that confound — stated
honestly above. The checkpoint with the best in-domain fit (printed by the
summary cell) is the one carried forward into MP2's alignment stage.

## 15. Try it yourself — interactive generation

The numbers in §14 are aggregate. This section loads the checkpoints and lets you
feed them **your own prompts**, inspecting generation quality directly.

Remember MP1 is a **continued-pretraining** model — a *text completer*, not a
chat model. Give it the **start** of a job posting and it continues; it will not
answer instruction-style prompts ("write a job description for a Java
developer") — that is MP2's job.

The logic lives in `src/generate_mp1.py`, which is also a standalone CLI:

```bash
.venv_atlm_pro/bin/python src/generate_mp1.py --model lora --prompt "We are looking for"
.venv_atlm_pro/bin/python src/generate_mp1.py --model all  --prompt "Senior Data Engineer"
.venv_atlm_pro/bin/python src/generate_mp1.py --model lora            # REPL
```

Decoding here uses **sampling** (temperature 0.8, top-p 0.95, repetition-penalty
1.3) rather than the greedy decoding of §14 — greedy maximises perplexity
comparability but reads worse.

In [ ]:
import sys

SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from generate_mp1 import load_model, generate

# Load all three checkpoints — the untrained base plus both adaptation methods —
# so generations can be compared the same three ways as the §14 perplexity table.
base_model, tokenizer = load_model("base", RUN)
full_model, _ = load_model("full", RUN)
lora_model, _ = load_model("lora", RUN)

MODELS = {"base": base_model, "full": full_model, "lora": lora_model}
print("loaded: base + full fine-tuned + LoRA-adapted SmolLM2-360M")

**Base vs full vs LoRA on the same prompts.** Each prompt is the opening of a
job posting — exactly the shape of text the model was continued-pretrained on.
The `base` row is the "before"; `full` and `lora` are the two trained variants.

In [ ]:
prompts = [
    "Senior Backend Engineer\n\nWe are looking for",
    "Responsibilities:\n-",
    "Required skills:",
]

for p in prompts:
    print("=" * 80)
    print(f"PROMPT: {p!r}")
    print("=" * 80)
    for name, model in MODELS.items():
        print(f"\n--- {name} ---")
        print(generate(model, tokenizer, p, max_new_tokens=100))
    print()

**Your turn.** Edit the prompt below and re-run — it generates from all three
checkpoints, so you see `base` ("before") against `full` and `lora`. Adjust
`temperature` / `max_new_tokens` to explore further.

In [ ]:
my_prompt = "Data Scientist with experience in"

for name, model in MODELS.items():
    print(f"--- {name} ---")
    print(generate(model, tokenizer, my_prompt, max_new_tokens=120, temperature=0.8))
    print()